# P300 Wave

## Overview
This notebook analyzes the P300 response from BNCI2014-009 (P300 speller). It compares averaged ERP for Target vs NonTarget stimuli at Fz, Cz, Pz.

## What to look for
- Positive peak appears ~300 ms after Target stimulus
- NonTarget response is much weaker

## 1. Install dependencies

In [ ]:
!pip install moabb mne scipy numpy plotly scikit-learn

## 2. Load data and average

In [ ]:
import numpy as np
from moabb.datasets import BNCI2014_009
from moabb.paradigms import P300

FS = 256
dataset = BNCI2014_009()
paradigm = P300(fmin=1, fmax=24)
X, labels, meta = paradigm.get_data(dataset=dataset, subjects=[1])

raw = dataset.get_data(subjects=[1])
s1 = raw[1]; sess = list(s1.values())[0]; run = list(sess.values())[0]
ch_names = run.ch_names
target_chs = ['Fz', 'Cz', 'Pz']
ch_idx = [ch_names.index(c) for c in target_chs]

target_avg = X[labels == 'Target'].mean(axis=0)
nontarget_avg = X[labels == 'NonTarget'].mean(axis=0)
t = np.arange(X.shape[2]) / FS * 1000
print(f"Data: {X.shape}, Targets: {np.sum(labels=='Target')}, NonTargets: {np.sum(labels=='NonTarget')}")

## 3. Interactive plot

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=3, subplot_titles=target_chs, shared_yaxes=True)
for i, (ch_name, ci) in enumerate(zip(target_chs, ch_idx)):
    fig.add_trace(go.Scatter(x=t, y=target_avg[ci], name='Target', line=dict(color='coral', width=2), showlegend=(i==0)), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=t, y=nontarget_avg[ci], name='NonTarget', line=dict(color='steelblue', width=1.5), showlegend=(i==0)), row=1, col=i+1)
    fig.add_vline(x=300, line_dash='dash', line_color='gray', row=1, col=i+1)
fig.update_layout(title='P300 ERP - BNCI2014-009 Subject 1', width=1200, height=400)
fig.update_xaxes(title_text='Time (ms)')
fig.update_yaxes(title_text='Amplitude (V)', row=1, col=1)
fig.show()

## Summary
- P300 is a positive wave ~300 ms after rare stimulus
- Most prominent at Cz and Pz
- Basis for spelling systems